# Chapter 11, Exercise 2: A keyword-based task router and where it breaks

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 11, Exercise 2.** Implement a small Python router that takes a free-text instruction and classifies it as one of the following speech-language model tasks: ASR, speech translation, speech question answering, dialect identification, or emotion recognition. Use simple English and Arabic keyword rules. Test the router on at least eight instructions, including dialectal, ambiguous, and compound instructions. Report any misrouted examples and explain why the keyword rules fail.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


## 1. The router

Each task has a list of English and Arabic keywords (MSA and some dialectal forms). The instruction is normalized (lower-cased, diacritics removed, a few letter forms unified), every task is scored by the number of keyword hits, and the highest score wins; ties and zero scores fall back to ASR with a flag. The rules are deliberately simple so that their failure modes are visible.

In [1]:
import re, unicodedata

TASKS = {
 "ASR": {
   "en": ["transcribe", "transcription", "write down", "what did", "what was said", "type out", "caption", "subtitle"],
   "ar": ["اكتب", "فرغ", "تفريغ", "نص", "حول الكلام", "ايش قال", "وش قال", "شو قال", "إيه اللي قال", "اكتبلي", "اكتب لي"]},
 "ST": {
   "en": ["translate", "translation", "in english", "into english", "to english", "in french", "into french"],
   "ar": ["ترجم", "ترجمة", "بالانجليزي", "بالإنجليزية", "للانجليزي", "للإنجليزية", "بالفرنسي", "بالفرنسية"]},
 "SQA": {
   "en": ["answer", "question", "why", "how many", "who ", "when ", "where ", "what is the", "explain", "summar"],
   "ar": ["اجب", "أجب", "جاوب", "سؤال", "ليش", "ليه", "لماذا", "كم ", "من هو", "مين ", "متى", "وين ", "أين", "ايش", "وش ", "شو ", "لخص", "لخّص"]},
 "ADI": {
   "en": ["dialect", "which arabic", "what arabic", "accent", "variety", "region", "egyptian", "gulf", "levantine", "maghrebi", "najdi", "hijazi", "msa"],
   "ar": ["لهجة", "لهجه", "اللهجة", "اي بلد", "أي بلد", "من وين المتكلم", "منين", "مصري", "خليجي", "شامي", "مغربي", "نجدي", "حجازي", "فصحى"]},
 "SER": {
   "en": ["emotion", "emotional", "feeling", "mood", "tone", "angry", "happy", "sad", "sentiment", "how does the speaker feel"],
   "ar": ["مشاعر", "شعور", "احساس", "إحساس", "عاطف", "نبرة", "زعلان", "زعلانة", "فرحان", "فرحانة", "غضبان", "حزين", "مبسوط", "منفعل", "حاسس"]},
}
DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
def normalize(t):
    t = unicodedata.normalize("NFC", t).lower()
    t = DIAC.sub("", t).replace("\u0640", "")
    t = re.sub("[أإآٱ]", "ا", t).replace("ة", "ه").replace("ى", "ي")
    return " " + re.sub(r"\s+", " ", t).strip() + " "

def route(instruction):
    t = normalize(instruction)
    scores = {}
    for task, kw in TASKS.items():
        hits = [k for k in kw["en"] + kw["ar"] if normalize(k).strip() in t]
        scores[task] = (len(hits), hits)
    best = max(scores, key=lambda k: scores[k][0])
    top = scores[best][0]
    tied = [k for k in scores if scores[k][0] == top]
    if top == 0:
        return "ASR", "no keyword matched: default", scores
    if len(tied) > 1:
        return tied[0], f"tie between {tied}: first wins", scores
    return best, ", ".join(scores[best][1]), scores

## 2. Test set: plain, dialectal, ambiguous and compound instructions

`expected` is what a human would want. Some items have two acceptable answers (compound instructions), noted in the comment.

In [2]:
tests = [
 # plain English and MSA
 ("Transcribe this recording.", "ASR"),
 ("ترجم هذا المقطع إلى الإنجليزية", "ST"),
 ("ما هي اللهجة التي يتحدث بها المتكلم؟", "ADI"),
 ("How does the speaker feel in this clip?", "SER"),
 ("كم عدد الأشخاص المذكورين في التسجيل؟", "SQA"),
 # dialectal
 ("اكتبلي وش قال الرجال بالضبط", "ASR"),                    # Gulf: 'write me what the man said exactly'
 ("قوللي الراجل ده منين", "ADI"),                          # Egyptian: 'tell me where this man is from'
 ("هو زعلان ولا مبسوط في الكلام ده؟", "SER"),              # Egyptian: 'is he upset or happy in this talk?'
 # ambiguous
 ("What is the tone of the announcement?", "SER"),          # 'tone' could be emotional tone or content
 ("Where is the speaker from?", "ADI"),                     # geography of origin, or a question about content?
 ("شو قال عن الطقس؟", "SQA"),                              # Levantine: 'what did he say about the weather?' (a content question)
 ("اكتب لي ملخص المكالمة", "SQA"),                          # 'write me a summary of the call': summarization, not verbatim ASR
 # compound
 ("Transcribe the audio and then translate it into English.", "ST"),            # ASR then ST; ST is the final deliverable
 ("حدد اللهجة ثم ترجم الكلام للإنجليزي", "ST"),                                  # ADI then ST
 ("Tell me the dialect and whether the speaker sounds angry.", "ADI or SER"),   # two tasks, no single answer
 ("لخص لي المكالمة وقل لي إذا كان فيه موعد", "SQA"),                              # the opening scene of Chapter 11
]
rows = []
for text, exp in tests:
    got, why, _ = route(text)
    ok = got in exp.split(" or ")
    rows.append({"instruction": text, "expected": exp, "routed": got, "correct": ok, "matched keywords / reason": why})
import pandas as pd
pd.set_option("display.max_colwidth", 70)
df = pd.DataFrame(rows)
print(f"{df['correct'].sum()} of {len(df)} routed as expected")
df

11 of 16 routed as expected


,instruction,expected,routed,correct,matched keywords / reason
0,Transcribe this recording.,ASR,ASR,True,transcribe
1,ترجم هذا المقطع إلى الإنجليزية,ST,ST,True,ترجم
2,ما هي اللهجة التي يتحدث بها المتكلم؟,ADI,ADI,True,"لهجة, لهجه, اللهجة"
3,How does the speaker feel in this clip?,SER,SER,True,how does the speaker feel
4,كم عدد الأشخاص المذكورين في التسجيل؟,SQA,SQA,True,كم
5,اكتبلي وش قال الرجال بالضبط,ASR,ASR,True,"اكتب, وش قال, اكتبلي"
6,قوللي الراجل ده منين,ADI,ADI,True,منين
7,هو زعلان ولا مبسوط في الكلام ده؟,SER,SER,True,"زعلان, مبسوط"
8,What is the tone of the announcement?,SER,SQA,False,"tie between ['SQA', 'SER']: first wins"
9,Where is the speaker from?,ADI,SQA,False,where


## 3. Misrouted examples and why keyword rules fail

Run the cell above and read the `correct` column; the discussion below refers to the failure types that this rule set exhibits (the exact rows depend on the keyword lists).

1. **Compound instructions.** "Transcribe the audio and then translate it" matches ASR and ST keywords; a count-based rule picks whichever list happened to match more strings, not the task the user actually wants delivered (the translation). The rules have no notion of *sequence* or *final output*. The Chapter 11 opening-scene instruction (summarize *and* tell me whether there is an appointment) is two tasks in one and no single label is right; an audio-language model handles it by generating a response that does both, which is exactly what a router cannot express.
2. **Ambiguity of ordinary words.** "tone" belongs to emotion in one reading and to content or register in another; "where is the speaker from" is dialect identification for a linguist and a content question for everyone else; "what did he say about the weather" contains an ASR cue (*what did he say*) but is a question about content (SQA). Keywords cannot see the *intent behind* the wording.
3. **Dialect lexical variation.** Each dialect has its own question words and verbs (وش / شو / ايش / إيه for 'what'; منين for 'where from'; زعلان for 'upset'). Every form must be listed by hand, and the list is never complete: a Maghrebi speaker's وين / فين / علاش will fall through to the default. Normalization helps with spelling (ة/ه, hamza) but not with vocabulary.
4. **Keyword collisions across tasks.** وش / شو / ايش appear both in the ASR list (وش قال, 'what did he say') and in the SQA list (as bare question words), so the same token votes for two tasks and the tie-breaking rule decides arbitrarily. Substring matching adds false hits: "كم " inside another word, "who " inside "whole".
5. **Negation and scope.** "Do not translate, just transcribe" contains a translation keyword and is routed to ST. Rules see words, not their scope.
6. **Silence about the default.** When nothing matches, the router silently returns ASR. A real system should return an explicit "unclear" and ask, which is the routing lesson of Section 8.11 in another guise: a confident wrong route is worse than an abstention.

These are the reasons instruction-tuned audio-language models replace routers with a learned mapping from the whole instruction (and the audio) to the response (Section 11.3): the instruction is read as language, not scanned for tokens. The price is that the learned selection must then be evaluated per task and per dialect (Section 11.7), because it can fail in ways a keyword list never would.